<div style="display: flex; justify-content: space-around; align-items: flex-start;">
  <div style="width: 100%; padding: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin: 10px;">
    <h2><h1 style="text-align: center; font-size: 4em; color: #46627F; margin-top: 0; margin-bottom: 0; line-height: 1;">Text Mining / Natural Language Processing </h1>
    <p>
<h1 style="text-align: center; color: #B1C0CF; margin-top: 0; margin-bottom: 0; line-height: 1;"> - STEP 2: Preprocessing and Modelling - </h1></p></h2>
      </div>
</div>

## LIBRARIES

In [1]:
# Standard libraries
import os

# Limit threads to avoid MKL/OpenMP issues on Windows
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"





In [2]:
import sys

from pathlib import Path

In [3]:

# Data manipulation
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt


In [4]:
# Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.model_selection import train_test_split



In [5]:
# Text processing / nlp
import re

import nltk
import nltk.corpus
import nltk.corpus
import nltk.tokenize
import spacy

In [6]:
# Load spaCy model
nlp_en = spacy.load("en_core_web_sm")
# nlp_pt = spacy.load("pt_core_news_sm") # from https://spacy.io/models/pt sm: small (12 MB), md: medium (40 MB), lg: large (541 MB)

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:
# Find the project root (assuming marker-based or script-relative path)
def find_project_root(marker="README.md"):
    current_dir = Path.cwd()
    while current_dir != current_dir.parent:  # Traverse up until root
        if (current_dir / marker).exists():
            return current_dir
        current_dir = current_dir.parent
    raise FileNotFoundError(f"Marker '{marker}' not found in any parent directory.")

project_root = find_project_root()
sys.path.append(str(project_root)) 

# Or use a relative path: project_root = Path(__file__).resolve().parent.parent
os.chdir(project_root)
print(f"Working directory set to: {project_root}")

Working directory set to: c:\Users\paulo\OneDrive\TRABALHO_SCRIPTS\textMining_v0


# LOAD DATA

In [ ]:
df_content = pd.read_excel(r'./data/input/PT2020-Lista-Operacoes_31-dezembro-2024.xlsx')

In [ ]:
df_content_clean = df_content[df_content['Resumo | Summary'] != '--']

# 0. Theoretical Background


**1. Latent Dirichlet Allocation (LDA)**


Latent Dirichlet Allocation (LDA) is a **probabilistic topic model**.  
It assumes each document is a mixture of hidden (latent) topics, and each topic is a distribution over words.

1.1 What LDA tries to do

Given only the observed **word counts** in documents, LDA tries to infer two hidden structures:

- **Topic–word distributions**  
  $ \beta_k(w) = P(w \mid \text{topic } k) $
- **Document–topic distributions**  
  $ \theta_d(k) = P(\text{topic } k \mid \text{document } d) $

The training process estimates matrices:

- Topic–word matrix $ \beta $
- Document–topic matrix $ \theta $

that best explain how the documents were generated.

1.2 The generative story

For each topic $k$:

1. Draw a distribution over words:  
   $$ \beta_k \sim \text{Dirichlet}(\eta) $$

For each document $d$:

1. Draw a distribution over topics:  
   $$ \theta_d \sim \text{Dirichlet}(\alpha) $$

2. For each word position $n$ in document $d$:
   - Choose a topic:  
     $$ z_{dn} \sim \text{Multinomial}(\theta_d) $$
   - Choose a word:  
     $$ w_{dn} \sim \text{Multinomial}(\beta_{z_{dn}}) $$

We only observe the words, not the latent structures; LDA infers them.

1.3 Why LDA is particular interesting for social scientists

- Topics are **probability distributions over words**.  
- Documents can contain **multiple themes**.  
- Hyperparameters allow control over topic sparsity and mixture characteristics.



**2. Non-Negative Matrix Factorization (NMF)**

NMF is a linear algebra approach, not probabilistic.

Given a non-negative matrix $X$ (e.g., TF–IDF):

$$ X \approx W H $$

- $W$ is the document–topic matrix  
- $H$ is the topic–word matrix  
- Both contain **only non-negative** values

2.1 Optimization objective

$$ \min_{W,H \ge 0} \| X - W H \|_F^2 $$

2.2 Why non-negativity matters

Because all values are non-negative:

- Topics become **additive combinations of words**
- Documents become **additive combinations of topics**

This produces very interpretable topics.

2.3 NMF vs LDA

| Aspect | LDA | NMF |
|-------|-----|------|
| Type | Probabilistic | Linear algebra |
| Input | Counts | TF-IDF or counts |
| Output | Probabilities | Non-negative weights |
| Interpretation | Very good | Very good |







**3. SVD and PCA**

**3.1 Singular Value Decomposition (SVD)**

SVD factorizes any matrix $X$:

$$ X = U \Sigma V^\top $$

Truncated SVD (keeping $k$ singular values):

$$ X \approx U_k \Sigma_k V_k^\top $$

This is the basis for **Latent Semantic Analysis (LSA)**.

**3.2 PCA**

PCA finds orthogonal directions that maximize variance.  
In text mining, PCA is implemented using **TruncatedSVD** because the data is sparse.

**3.3 Comparison**

| Property | SVD/PCA | NMF | LDA |
|----------|---------|-----|-----|
| Loadings | Positive & negative | Non-negative | Probabilities |
| Interpretability | Medium | High | High |
| Main use | Dimensionality reduction | Topic discovery | Topic discovery |







**4. When to Use What**

**4.1 LDA**
- Best when wanting a **generative story**
- Provides $P(\text{topic}|\text{document})$

**4.2 NMF**
- Best for **clean, interpretable topics**
- Works very well with TF-IDF

**4.3 SVD/PCA**
- Use for **dimensionality reduction**, document embeddings, or similarity search



# 1. Classical Text Mining Pipeline

Goal: Obtain a topic map of the corpus using TF-IDF and LDA/NMF.



Apply basic text preprocessing conceots:
- Tokenization: splitting text into words/tokens.
- Stopwords: common words with little semantic content (“the”, “and”, …).
- Lemmatization vs stemming: map words to a base form:
    - Stemming: crude, rule-based cutting (e.g. “studies”, “studying” → “studi”).
    - Lemmatization: uses vocabulary and grammar (“studies” → “study”).

For the classical phase, we can start with simple, built-in tokenization and stopword lists from scikit-learn.

This is the classical pipeline:
BoW → TF-IDF / Counts → LDA → interpret topics.

## 1.1. Basic preprocessing

In [ ]:
# 1.1 Define a simple TF-IDF vectorizer with built-in preprocessing
tfidf_vectorizer = TfidfVectorizer(
    input='content',
    lowercase=True,
    stop_words='english',  # built-in English stopword list
    ngram_range=(1, 2),    # unigrams + bigrams
    min_df=5,              # ignore very rare terms
    max_df=0.7             # ignore too frequent terms
)

# Fit and transform abstracts
tfidf_matrix = tfidf_vectorizer.fit_transform(df["abstract"])

tfidf_matrix.shape

## 1.2 Topic Modeling with LDA or NMF 

In [ ]:
# 1.2.1 Create Count matrix for LDA
count_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7
)

count_matrix = count_vectorizer.fit_transform(df["abstract"])

count_matrix.shape

LDA is a **probabilistic topic model**. It assumes:

- Each document is a **mixture of topics**.
- Each topic is a **probability distribution over words**.

Mathematically:
- For each document, we draw a distribution over topics.
- For each word in the document, we pick a topic and then a word from that topic.
- We observe only words; the topics and topic proportions are **latent** and must be inferred.

You can think of topics as **latent themes** in the corpus (e.g., "urban mobility", "renewable energy", "social policy").

We will:

1. Fit an LDA model on the **count matrix**.
2. Inspect the **top words** per topic.
3. Look at **which topic is dominant in each document**.

In practice we must **choose the number of topics** (`n_components`). This is a modelling choice and can be evaluated later.

In [ ]:
# 1.2.2 Fit LDA
n_topics = 10  # choose a number, can be tuned

lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method='batch'
)
lda_doc_topic = lda.fit_transform(count_matrix)

## 1.3. Inspecting Topics

Each topic is a distribution over terms (weights in model.components_).

Top weighted terms help us interpret the topic

In [ ]:
def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic #{topic_idx}")
        # larger weight = more important term in the topic
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_terms = [feature_names[i] for i in top_indices]
        print("  " + ", ".join(top_terms))
        print()

count_feature_names = count_vectorizer.get_feature_names_out()
print_top_words(lda, count_feature_names, n_top_words=10)

## 1.4. Document–Topic Assignments

which topic dominates each abstract?


The `lda_doc_topic` matrix contains, for each document, the **probability of belonging to each topic**.

- Row = document
- Column = topic
- Entry = P(topic | document)

We can:

- For each document, find the **dominant topic** (the topic with highest probability).
- For each topic, look at a few representative documents.

This helps us interpret each topic as a **research theme**.

In [ ]:
# For each document, the topic with highest probability
df["dominant_topic_lda"] = lda_doc_topic.argmax(axis=1)

df[["title", "dominant_topic_lda"]].head()

In [ ]:
def show_docs_for_topic(df, topic_id, n_docs=3):
    subset = df[df["dominant_topic_lda"] == topic_id].head(n_docs)
    for i, row in subset.iterrows():
        print(f"\n--- Document {i} ---")
        print("Title:", row["title"])
        print("Abstract:", row["abstract"][:600], "...")
        
show_docs_for_topic(df, topic_id=0, n_docs=3)

# 2. spaCy-Enhanced Pipeline

Now we add linguistic knowledge:

- Better lemmatization
  
- POS filtering

- Noun chunks

- (Optionally NER, dependency parsing)

The classical approach treats text as an **unordered bag of words**. It does not know:

- which words are nouns vs verbs vs adjectives
- that "policies" and "policy" are the same lemma
- multi-word expressions like "climate change" as a single concept

To improve this, we use **spaCy**, a modern Natural Language Processing (NLP) library.

spaCy can provide:

- **Tokenization**: splitting text into tokens.
- **Part-of-Speech (POS) tagging**: NOUN, VERB, ADJ, etc.
- **Lemmatization**: converting "studies", "studying" → "study".
- **Noun chunks**: base noun phrases like "energy transition", "social housing".
- **Named Entity Recognition (NER)**: detecting entities like countries, organizations, etc. (used later for interpretation).

We will:

1. Use spaCy to build a **cleaned, lemmatized text** for each document.
2. Filter tokens by POS (keeping mainly content words like NOUN, PROPN, ADJ).
3. Run TF-IDF + NMF or LDA on this spaCy-enhanced corpus.

## 2.1. Load spaCy and Language Model

In [ ]:

# You may need: python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # enable others later if needed
# disable parser/NER initially for speed while doing lemmatization & POS.

## 2.2. spaCy-based Preprocessing Function

Idea:

Use spaCy to:

- tokenize

- lemmatize

- filter by POS

- drop stopwords and punctuation

We’ll keep mainly NOUN, PROPN, ADJ, and maybe VERB (you can tweak).



Mechanics / principles:

- We are now forming a vocabulary of lemmas (base forms), not surface forms.

- We preserve grammatically important words (nouns, proper nouns, adjectives).

- We remove high-frequency “function words” more robustly via token.is_stop and POS.

»»» This should create cleaner and more semantically focused features for topic modeling.

In [ ]:
# 2.2 spaCy-based tokenizer/preprocessor

allowed_postags = {"NOUN", "PROPN", "ADJ"}  # optionally add "VERB"

def spacy_preprocess(text):
    doc = nlp(text)
    tokens = []
    for token in doc:
        # Skip stopwords, punctuation, spaces
        if token.is_stop or token.is_punct or token.is_space:
            continue
        
        # Filter by POS
        if token.pos_ not in allowed_postags:
            continue
        
        # Use lemma (lowercased) as feature
        lemma = token.lemma_.lower()
        
        # Filter out very short lemmas
        if len(lemma) < 3:
            continue
        
        tokens.append(lemma)
        
    return " ".join(tokens)

# Test on a single abstract
print(df["abstract"].iloc[0][:300])
print("---")
print(spacy_preprocess(df["abstract"].iloc[0])[:300])


## 2.3. Apply spaCy Preprocessing to All docs

In [ ]:
# This can take some time on large corpora
df["spacy_clean"] = df["abstract"].astype(str).apply(spacy_preprocess)

df[["abstract", "spacy_clean"]].head()


Now, redo the feature extraction & topic modeling, but on spacy_clean

## 2.4. TF-IDF + NMF on spaCy-cleaned Text

Now we build a new TF-IDF matrix using the spaCy-cleaned text.
This matrix should have:
- lemmas instead of raw word forms,
- mainly content words,
- potentially less noise and sparsity.

We then fit an **NMF (Non-Negative Matrix Factorization)** topic model.

**Concept: NMF**

- Given a non-negative matrix (e.g., TF-IDF), NMF finds two smaller non-negative matrices:
  - Document-topic matrix
  - Topic-term matrix
- It is an algebraic factorization, not a generative probabilistic model like LDA.
- Topics are often quite interpretable when used with TF-IDF.

In [ ]:
# 2.4 TF-IDF on spaCy preprocessed text
tfidf_vectorizer_spacy = TfidfVectorizer(
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)
)

tfidf_matrix_spacy = tfidf_vectorizer_spacy.fit_transform(df["spacy_clean"])

tfidf_matrix_spacy.shape


In [ ]:
# 2.4.2 NMF topic model
n_topics_spacy = 10  # can differ from LDA choice

nmf = NMF(
    n_components=n_topics_spacy,
    random_state=42,
    init='nndsvd',
    max_iter=500
)

nmf_doc_topic = nmf.fit_transform(tfidf_matrix_spacy)


## 2.5. Inspect spaCy-Enhanced Topics

In [ ]:
def print_top_words_nmf(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic #{topic_idx}")
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_terms = [feature_names[i] for i in top_indices]
        print("  " + ", ".join(top_terms))
        print()

tfidf_feature_names_spacy = tfidf_vectorizer_spacy.get_feature_names_out()
print_top_words_nmf(nmf, tfidf_feature_names_spacy, n_top_words=10)


Are topics more coherent now?

Do noun phrases or adjectives help interpret?

## 2.6. Using Noun Chunks as Features (Optional)

We can also define features at the phrase level using noun chunks: “energy transition”, “climate change”, “social inequality”

Use a function to extract noun chunks and build a separate representation, or simply append them.


In [ ]:
# Re-enable parser for noun chunks
nlp_chunk = spacy.load("en_core_web_sm", disable=["ner"])

def extract_noun_chunks(text):
    doc = nlp_chunk(text)
    chunks = []
    for chunk in doc.noun_chunks:
        # Clean up chunk text
        chunk_text = chunk.text.strip().lower()
        if len(chunk_text.split()) > 1:  # only multi-word chunks
            chunks.append("_".join(chunk_text.split()))  # e.g. "climate_change"
    return chunks

# Example
print(extract_noun_chunks(df["abstract"].iloc[0])[:10])


In [ ]:
def spacy_preprocess_with_chunks(text):
    # Word-level processing as before
    doc = nlp(text)
    tokens = []
    for token in doc:
        if token.is_stop or token.is_punct or token.is_space:
            continue
        if token.pos_ not in allowed_postags:
            continue
        lemma = token.lemma_.lower()
        if len(lemma) < 3:
            continue
        tokens.append(lemma)
    
    # Add noun chunks
    doc2 = nlp_chunk(text)
    chunks = []
    for chunk in doc2.noun_chunks:
        chunk_text = chunk.text.strip().lower()
        if len(chunk_text.split()) > 1:
            chunks.append("_".join(chunk_text.split()))
    
    # Combine tokens and chunks
    all_terms = tokens + chunks
    return " ".join(all_terms)

df["spacy_chunks"] = df["abstract"].astype(str).apply(spacy_preprocess_with_chunks)
df[["spacy_clean", "spacy_chunks"]].head()


repeat TF-IDF + NMF or LDA with spacy_chunks as input, and see whether topics become even more interpretable.

## 2.7. Named Entity Recognition (NER) for Topic Context

In [ ]:
# Load full pipeline with NER
nlp_ner = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp_ner(text)
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    return ents

# Test
extract_entities(df["abstract"].iloc[0])


focus on certain entity types, e.g. GPE (countries/regions), ORG (organizations).

In [ ]:
from collections import Counter

def count_entities(df, label_filter=None, n_top=20):
    counter = Counter()
    for text in df["abstract"]:
        doc = nlp_ner(text)
        for ent in doc.ents:
            if label_filter and ent.label_ not in label_filter:
                continue
            counter[ent.text] += 1
    return counter.most_common(n_top)

# Most common countries/places
top_gpe = count_entities(df, label_filter={"GPE"})
top_gpe[:20]


Cross-tab topics with entities: which entities are prominent in which topics?

In [ ]:
# Assign dominant NMF topic
df["dominant_topic_nmf"] = nmf_doc_topic.argmax(axis=1)

# Example: which countries are most frequent in topic 0?
topic_id = 0
subset = df[df["dominant_topic_nmf"] == topic_id]

count_entities(subset, label_filter={"GPE"}, n_top=10)


# 3. Optional: LLM Summarization on Top of the Analysis

The idea here is **not** to replace everything with a Large Language Model (LLM),
 but to use an LLM as a **summarization layer on top of our structured analysis**.

Workflow idea:

1. For each topic (e.g., NMF topic from spaCy TF-IDF), collect:
   - The top 10–20 words.
   - A few representative abstracts (documents with high topic probability).
2. Build a **prompt** describing the topic with:
   - "Here are the top words..."
   - "Here are some example abstracts..."
3. Ask the LLM:
   - "Give a short label for this topic and summarize it in a few sentences."

This can help write **human-readable summaries** of research themes.
However, it is important to **critically evaluate** the LLM’s output:

- Does it stay close to the evidence?
- Does it invent claims not supported by the abstracts?
- How does its summary compare to your own interpretation?

## 3.1. Prepare Topic Descriptions

In [ ]:
# Build data structure with topic info
topics_info = []

for topic_idx in range(n_topics_spacy):
    # Top words for topic
    topic = nmf.components_[topic_idx]
    top_indices = topic.argsort()[::-1][:10]
    top_terms = [tfidf_feature_names_spacy[i] for i in top_indices]
    
    # Sample abstracts
    topic_docs_idx = np.argsort(-nmf_doc_topic[:, topic_idx])[:3]
    sample_abstracts = df["abstract"].iloc[topic_docs_idx].tolist()
    
    topics_info.append({
        "topic_id": topic_idx,
        "top_terms": top_terms,
        "sample_abstracts": sample_abstracts
    })

topics_info[0]


## 3.2. Example Prompt for a Local LLM

In [ ]:

import textwrap

def build_prompt_for_topic(topic_id, model, feature_names, doc_topic_matrix, df, n_top_terms=10, n_docs=3):
    """
    Build a prompt string for a given topic:
    - top words from the model
    - a few representative abstracts
    """
    # top words
    topic = model.components_[topic_id]
    top_indices = topic.argsort()[::-1][:n_top_terms]
    top_terms = [feature_names[i] for i in top_indices]
    terms_str = ", ".join(top_terms)

    # representative docs (with highest topic weights)
    doc_indices = np.argsort(-doc_topic_matrix[:, topic_id])[:n_docs]
    abstracts_text = "\n\n".join(
        f"Abstract {i+1}:\n{df['abstract'].iloc[idx]}"
        for i, idx in enumerate(doc_indices)
    )

    prompt = f"""
    You are helping a social science researcher understand topics in a corpus of academic abstracts.

    We have identified Topic {topic_id} with the following top keywords:
    {terms_str}

    Here are a few representative abstracts for this topic:
    {abstracts_text}

    Task:
    1. Provide a concise label (a short phrase) that captures the main theme of Topic {topic_id}.
    2. Write a short paragraph (4-6 sentences) summarizing what this research topic is about.
    3. Mention any recurring methods, populations, or geographic regions if you can infer them.

    Answer in plain English.
    """
    return textwrap.dedent(prompt).strip()

# Example: build a prompt for NMF topic 0
example_prompt = build_prompt_for_topic(
    topic_id=0,
    model=nmf,
    feature_names=tfidf_feature_names_spacy,
    doc_topic_matrix=nmf_doc_topic,
    df=df,
    n_top_terms=10,
    n_docs=3
)
print(example_prompt[:1000], "...")


In [ ]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/chat"  # default Ollama HTTP endpoint
OLLAMA_MODEL = "llama3"  # change to whatever model you use in `ollama run`


def call_ollama_llm(prompt, model=OLLAMA_MODEL, temperature=0.4):
    """
    Call a local LLM served by Ollama via its HTTP /api/chat endpoint.

    Parameters
    ----------
    prompt : str
        The user prompt (e.g., the topic summary prompt we built).
    model : str
        The Ollama model name (e.g., 'llama3', 'mistral', etc.).
    temperature : float
        Sampling temperature (higher = more creative, lower = more deterministic).

    Returns
    -------
    str
        The assistant's reply text.
    """
    url = OLLAMA_URL
    headers = {"Content-Type": "application/json"}
    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {
            "temperature": temperature
        },
        # For notebooks, it's often easier to disable streaming
        "stream": False
    }

    response = requests.post(url, headers=headers, data=json.dumps(payload))
    response.raise_for_status()  # raise error if request failed

    data = response.json()

    # Ollama /api/chat returns either:
    # - a single 'message' when stream=False, or
    # - streaming chunks when stream=True (we disabled that)
    if "message" in data and "content" in data["message"]:
        return data["message"]["content"]
    else:
        # Fallback: inspect full JSON if structure differs
        return json.dumps(data, indent=2)


In [ ]:
# Build prompt for topic 0 (NMF on spaCy TF-IDF)
topic_id = 0
prompt = build_prompt_for_topic(
    topic_id=topic_id,
    model=nmf,
    feature_names=tfidf_feature_names_spacy,
    doc_topic_matrix=nmf_doc_topic,
    df=df,
    n_top_terms=10,
    n_docs=3
)

print("PROMPT (first 800 chars):\n")
print(prompt[:800], "...\n")

# Call Ollama
summary = call_ollama_llm(prompt)
print("LLM SUMMARY:\n")
print(summary)

In [ ]:
topic_summaries = {}

for t in range(n_topics_nmf):
    print(f"\n===== TOPIC {t} =====")
    prompt = build_prompt_for_topic(
        topic_id=t,
        model=nmf,
        feature_names=tfidf_feature_names_spacy,
        doc_topic_matrix=nmf_doc_topic,
        df=df,
        n_top_terms=10,
        n_docs=3
    )
    summary = call_ollama_llm(prompt)
    topic_summaries[t] = summary
    print(summary)

# 4. Evaluating Topic Models



Topic modelling is unsupervised: there is no "ground truth" topics.
So, we must evaluate topic models using **indirect criteria**:

1. **Interpretability of topic-word lists**:
    - Are the top 10–20 words per topic coherent and meaningful?
    - Do they suggest a clear theme?

2. **Topic size distribution**:
    - Are some topics assigned to almost no documents?
    - Are some topics "catch-all" dominating almost everything?

3. **Stability across runs**:
    - If we run the model again with a different random seed, do we get similar topics?
    - Large differences suggest unstable or poorly specified models.

4. **Quantitative coherence metrics (optional)**:
   - Metrics like **c_v**, **u_mass** etc. try to measure how often the top words of a topic co-occur.
   - Implemented in libraries like `gensim`.

## 5.1 Topic Size Distribution

We start with a very simple diagnostic:

 - For each model, count how many documents have each topic as dominant.

If a topic has **very few documents**, it might be:
- a very specific niche topic (could be fine),
- or an unstable / noisy topic (especially if its top words look incoherent).

If a topic has **most documents**, it might be absorbing multiple themes, which is not ideal.

In [ ]:
# LDA topic sizes
lda_topic_counts = df["dominant_topic_lda"].value_counts().sort_index()

plt.bar(lda_topic_counts.index, lda_topic_counts.values)
plt.xlabel("LDA Topic")
plt.ylabel("Number of documents")
plt.title("LDA: Topic Size Distribution")
plt.show()

lda_topic_counts


In [ ]:

# NMF topic sizes
nmf_topic_counts = df["dominant_topic_nmf"].value_counts().sort_index()

plt.bar(nmf_topic_counts.index, nmf_topic_counts.values)
plt.xlabel("NMF Topic")
plt.ylabel("Number of documents")
plt.title("NMF: Topic Size Distribution (spaCy-cleaned)")
plt.show()

nmf_topic_counts


## 4.2 Topic Stability Across Random Seeds (Simple Approximation)

A simple (approximate) way to check **stability**:

 1. Fit the same model (e.g., NMF on spaCy TF-IDF) twice with different random seeds.
 2. For each topic in model A, find the "most similar" topic in model B by:
    - comparing the sets of top words,
    - computing overlap (e.g., how many top words are shared).

 A very unstable model will produce very different top word lists in different runs.

 Below we show a simple procedure for NMF on the spaCy TF-IDF matrix.

In [ ]:

def get_top_terms_per_topic(model, feature_names, n_top=20):
    """
    Return a list of lists: top terms for each topic.
    """
    all_topics = []
    for topic in model.components_:
        top_indices = topic.argsort()[::-1][:n_top]
        top_terms = [feature_names[i] for i in top_indices]
        all_topics.append(top_terms)
    return all_topics

# Fit a second NMF model with a different random_state
nmf2 = NMF(
    n_components=n_topics_nmf,
    random_state=123,
    init="nndsvd",
    max_iter=500
)
nmf2_doc_topic = nmf2.fit_transform(tfidf_matrix_spacy)

# Get top terms for both models
topics_nmf1 = get_top_terms_per_topic(nmf, tfidf_feature_names_spacy, n_top=20)
topics_nmf2 = get_top_terms_per_topic(nmf2, tfidf_feature_names_spacy, n_top=20)


In [ ]:

def topic_overlap_score(topic_terms_a, topic_terms_b):
    """
    Simple score: fraction of shared terms between two topics.
    """
    set_a = set(topic_terms_a)
    set_b = set(topic_terms_b)
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

# For each topic in model 1, find the best match in model 2
overlap_matrix = np.zeros((n_topics_nmf, n_topics_nmf))

for i in range(n_topics_nmf):
    for j in range(n_topics_nmf):
        overlap_matrix[i, j] = topic_overlap_score(topics_nmf1[i], topics_nmf2[j])

overlap_matrix


In [ ]:

# For each topic in model 1, find best overlapping topic in model 2
best_matches = overlap_matrix.argmax(axis=1)
best_scores = overlap_matrix.max(axis=1)

for i in range(n_topics_nmf):
    print(f"Topic {i} (model 1) best matches Topic {best_matches[i]} (model 2) "
          f"with overlap score {best_scores[i]:.2f}")


**Interpretation of overlap scores:**

 - Scores near 1.0: the top terms sets are very similar → topic is stable across runs.
 - Scores near 0: very little overlap → topic is unstable or not well-identified.

 This is a *simple heuristic*, not a formal statistical test, but it helps build intuition.

 You could repeat this with LDA (on the count matrix) in exactly the same way.

## 4.3 Optional: Topic Coherence with `gensim`